In [1]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

panel = pd.read_parquet("Panel.parquet")

HARX = ["RV_daily", "RV_weekly", "RV_monthly",
        "CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]
CREDIT = ["CDS_level", "CDS_change_5d"]

FEAT_LABELS = {"RV_daily": "RV daily", "RV_weekly": "RV weekly", "RV_monthly": "RV monthly",
               "CDS_level": "CDS level", "CDS_change_5d": "CDS 5d chg",
               "VIX": "VIX", "Yield_slope": "Yield slope", "BAA10Y": "Baa spread"}

SEC_ORDER = ["XLF", "XLY", "XLE", "XLI", "XLK", "XLV"]   # high-leverage first

RF_PARAMS  = dict(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
XGB_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                  subsample=0.8, random_state=42, verbosity=0)

print(f"Loaded {len(panel):,} rows, {panel['Firm'].nunique()} firms")

Loaded 211,196 rows, 56 firms


In [ ]:
# Models trained on pre-2020 window; SHAP computed on out-of-sample period.
tgt = "Target_h5"
d  = panel.dropna(subset=HARX + [tgt]).copy()
tr = d[d["Date"].dt.year <  2020]
te = d[d["Date"].dt.year >= 2020].copy()
Xtr, ytr = tr[HARX].values, tr[tgt].values

print("Training models for SHAP...")
rf = RandomForestRegressor(**RF_PARAMS).fit(Xtr, ytr)
xg = xgb.XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)

# Sample 8,000 out-of-sample rows for tractable exact TreeSHAP.
samp = te.sample(min(8000, len(te)), random_state=42)
Xs = samp[HARX].values

print("Computing SHAP values (both models)...")
rf_sv = shap.TreeExplainer(rf).shap_values(Xs)
xg_sv = shap.TreeExplainer(xg).shap_values(Xs)

Training models for SHAP...
Computing SHAP values (both models)...


In [ ]:
def credit_share_by_sector(sv, samp):
    # Mean |SHAP| of credit features as a percentage of total mean |SHAP|, per sector.
    abs_sv = pd.DataFrame(np.abs(sv), columns=HARX)
    abs_sv["Sector"] = samp["Sector"].values
    out = {}
    for sec in SEC_ORDER:
        sub = abs_sv[abs_sv["Sector"] == sec]
        total  = sub[HARX].mean().sum()
        credit = sub[CREDIT].mean().sum()
        out[sec] = 100 * credit / total
    return out
    
rf_share = credit_share_by_sector(rf_sv, samp)
xg_share = credit_share_by_sector(xg_sv, samp)

credit_share = pd.DataFrame({
    "Sector": SEC_ORDER,
    "Credit_pct_RF":  [round(rf_share[s], 1) for s in SEC_ORDER],
    "Credit_pct_XGB": [round(xg_share[s], 1) for s in SEC_ORDER],
})
credit_share.to_csv("SHAP_Credit_Share.csv", index=False)

print("Credit features as % of total SHAP importance, by sector (one-week horizon):\n")
print(credit_share.to_string(index=False))

In [ ]:
def sector_feature_table(sv, samp, title):
    abs_sv = pd.DataFrame(np.abs(sv), columns=HARX)
    abs_sv["Sector"] = samp["Sector"].values
    rows = []
    for sec in SEC_ORDER:
        sub = abs_sv[abs_sv["Sector"] == sec]
        rows.append({"Sector": sec, **{FEAT_LABELS[f]: round(sub[f].mean(), 4) for f in HARX}})
    df = pd.DataFrame(rows)
    print(f"\n{title} — mean |SHAP| by sector:")
    print(df.to_string(index=False))
    return df

rf_detail  = sector_feature_table(rf_sv, samp, "Random Forest")
xgb_detail = sector_feature_table(xg_sv, samp, "XGBoost")

rf_detail.to_csv("SHAP_Detail_RF.csv", index=False)
xgb_detail.to_csv("SHAP_Detail_XGB.csv", index=False)